# LLM VRAM 계산과 T4 실측

가중치와 KV 캐시 크기를 계산하고, Hugging Face 저장 용량과 Colab T4의 실제 메모리 할당을 대조합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/llm_vram_calculator.ipynb)

1~2절은 CPU 런타임으로 충분합니다. 3절은 Colab에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. GB는 10^9 bytes입니다.

In [ ]:
from math import ceil

GB = 10**9


def weight_bytes(params, bytes_per_value):
    return params * bytes_per_value


def kv_bytes(layers, kv_heads, head_dim, bytes, tokens, batch=1):
    # K와 V를 모두 저장하므로 2를 곱한다.
    return 2 * layers * kv_heads * head_dim * bytes * tokens * batch


def nvfp4_bytes(fp4_values):
    # FP4 값은 각각 0.5바이트이고, 16개 블록마다 FP8 스케일 1바이트를 더한다.
    return fp4_values * 0.5 + ceil(fp4_values / 16)

## 1. Hugging Face 저장 용량과 대조

공개 모델 API에서 정밀도별 파라미터 수와 저장소 파일 용량(`usedStorage`)을 읽습니다. `REPOS`의 저장소 이름을 바꿔 다른 모델을 비교할 수 있습니다.

In [ ]:
import requests

# Muse 가중치 라이선스는 각 저장소의 모델 카드와 LICENSE 참조.
# 이 셀은 Hugging Face API의 공개 메타데이터만 조회한다.
REPOS = (
    "meta-models/Muse-Spark-Open-816B-A42B",
    "meta-models/Muse-Spark-Open-816B-A42B-NVFP4",
)
BYTES_PER_DTYPE = {
    "BOOL": 1, "U8": 1, "I8": 1, "F8": 1, "F8_E4M3": 1, "F8_E5M2": 1,
    "U16": 2, "I16": 2, "BF16": 2, "F16": 2,
    "U32": 4, "I32": 4, "F32": 4, "U64": 8, "I64": 8, "F64": 8,
}
NVFP4_REPO = "meta-models/Muse-Spark-Open-816B-A42B-NVFP4"


def compare_storage(session, repo):
    response = session.get(
        f"https://huggingface.co/api/models/{repo}",
        params=[("expand[]", "safetensors"), ("expand[]", "usedStorage")],
        timeout=20,
    )
    response.raise_for_status()
    data = response.json()
    counts = data["safetensors"]["parameters"]
    stored = data["usedStorage"]
    if not isinstance(counts, dict) or not counts or type(stored) is not int or stored <= 0:
        raise ValueError("파라미터 수 또는 usedStorage 형식이 예상과 다릅니다.")
    if any(type(count) is not int or count < 0 for count in counts.values()):
        raise ValueError("정밀도별 파라미터 수가 음이 아닌 정수가 아닙니다.")
    if repo == NVFP4_REPO and "U8" not in counts:
        raise ValueError("Muse NVFP4 응답에 U8 항목이 없습니다.")

    sizes = {}
    for dtype, count in counts.items():
        if dtype == "U8" and repo == NVFP4_REPO:
            # Muse NVFP4의 U8 수는 원본 실험과 같이 논리 FP4 값 개수로 해석한다.
            sizes["U8 (논리 FP4 값 + FP8 스케일)"] = nvfp4_bytes(count)
        else:
            sizes[dtype] = weight_bytes(count, BYTES_PER_DTYPE[dtype])

    print(f"\n{repo}")
    for count, (label, size) in zip(counts.values(), sizes.items()):
        print(f"  {label}: {count:,}개 → {size:,.1f} bytes")
    calculated = sum(sizes.values())
    difference = (calculated - stored) / stored
    print(f"가중치 계산: {calculated:,.1f} bytes ({calculated / GB:,.6f} GB)")
    print(f"usedStorage: {stored:,} bytes ({stored / GB:,.6f} GB)")
    print(f"저장 용량 대비 차이: {difference:+.6%} | 1% 이내: {'예' if abs(difference) <= 0.01 else '아니요'}")
    return calculated


weight_estimates = {}
with requests.Session() as session:
    session.trust_env = False  # 환경과 .netrc에 저장된 인증 정보를 사용하지 않는다.
    for repo in REPOS:
        try:
            weight_estimates[repo] = compare_storage(session, repo)
        except requests.RequestException as error:
            print(f"{repo}: 모델 API 요청 실패 ({error})")
        except (KeyError, TypeError, ValueError) as error:
            print(f"{repo}: 모델 API 응답을 계산할 수 없습니다 ({error})")

## 2. Muse Spark Open의 KV 캐시

[Muse Spark Open 모델 카드](https://huggingface.co/meta-models/Muse-Spark-Open-816B-A42B)의 층 수, KV 헤드 수, 헤드 차원에 BF16을 적용합니다. 32K, 128K, 1M은 각각 32,768, 131,072, 1,048,576토큰입니다.

In [ ]:
LAYERS, KV_HEADS, HEAD_DIM, BF16_BYTES = 62, 16, 64, 2
CONTEXTS = {"32K": 32_768, "128K": 131_072, "1M": 1_048_576}
MUSE_REPOS = ("meta-models/Muse-Spark-Open-816B-A42B", NVFP4_REPO)

# 배치 1에서 모든 층이 전체 문맥을 보는 경우의 KV 캐시 상한이다.
bytes_per_token = kv_bytes(LAYERS, KV_HEADS, HEAD_DIM, BF16_BYTES, tokens=1)
print(f"BF16 KV 캐시: {bytes_per_token:,} bytes/token")
print("문맥	KV (GB)	BF16판+KV (GB)	NVFP4판+KV (GB)")
for label, tokens in CONTEXTS.items():
    cache = bytes_per_token * tokens
    totals = [
        f"{(weight_estimates[repo] + cache) / GB:,.3f}" if repo in weight_estimates else "—"
        for repo in MUSE_REPOS
    ]
    print(f"{label}	{cache / GB:,.3f}	" + "	".join(totals))

if any(repo not in weight_estimates for repo in MUSE_REPOS):
    print("—: 1절에서 해당 Muse 가중치 계산값을 얻지 못했습니다.")

print("\n128K 기준 기기 수량 (메모리를 합산할 수 있다고 가정한 이상적 하한)")
print("기기	BF16판+KV	NVFP4판+KV")
cache_128k = bytes_per_token * CONTEXTS["128K"]
for device, capacity_gb in (("H100 80GB", 80), ("H200 141GB", 141), ("Mac Studio 512GB", 512)):
    counts = [
        str(ceil((weight_estimates[repo] + cache_128k) / (capacity_gb * GB)))
        if repo in weight_estimates else "—"
        for repo in MUSE_REPOS
    ]
    print(f"{device}	" + "	".join(counts))
print("실행 버퍼, 분산 통신, 운영체제 메모리는 계산에서 제외합니다.")

## 3. Colab T4에서 Qwen2.5 실측

[Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)와 [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)의 라이선스는 Apache-2.0입니다. `MODEL_REPO`로 모델을 고릅니다. 배치 1, FP16, SDPA에서 길이와 로짓 조건마다 새 KV 캐시로 prefill을 한 번 실행합니다.

In [ ]:
import subprocess
import sys

import torch

if torch.cuda.is_available():
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "transformers==5.22.1", "accelerate",
    ])
else:
    print("CUDA가 없습니다. Colab에서 T4 GPU 런타임을 선택하면 실측할 수 있습니다.")

In [ ]:
import gc

MODEL_REPO = "Qwen/Qwen2.5-0.5B-Instruct"
# MODEL_REPO = "Qwen/Qwen2.5-1.5B-Instruct"
# 모델 출처: 위 Hugging Face 저장소, 라이선스: Apache-2.0
# 파라미터 수 출처: 각 저장소의 Hugging Face 모델 API safetensors.total
API_PARAMETERS = {
    "Qwen/Qwen2.5-0.5B-Instruct": 494_032_768,
    "Qwen/Qwen2.5-1.5B-Instruct": 1_543_714_304,
}
LENGTHS = (1024, 4096, 16384)
BATCH_SIZE, FP16_BYTES = 1, 2


def print_bytes(label, size):
    value = "측정 불가" if size is None else f"{size:,} bytes ({size / GB:,.3f} GB)"
    print(f"{label}: {value}")


def tensor_bytes(tensor):
    return tensor.numel() * tensor.element_size()


def cache_tensor_bytes(cache):
    if hasattr(cache, "layers"):
        pairs = ((layer.keys, layer.values) for layer in cache.layers)
    elif hasattr(cache, "key_cache"):
        pairs = zip(cache.key_cache, cache.value_cache)
    else:
        pairs = cache
    return sum(tensor_bytes(tensor) for pair in pairs for tensor in pair)


def cleanup(device):
    torch.cuda.synchronize(device)
    gc.collect()
    torch.cuda.empty_cache()


@torch.inference_mode()
def measure_prefill(model, device, length, logits_to_keep, baseline):
    config = model.config
    generator = torch.Generator(device="cpu").manual_seed(length)
    ids = torch.randint(config.vocab_size, (BATCH_SIZE, length), generator=generator, device="cpu")
    torch.cuda.synchronize(device)
    torch.cuda.reset_peak_memory_stats(device)
    ids = ids.to(device)
    output = model(input_ids=ids, use_cache=True, logits_to_keep=logits_to_keep)
    torch.cuda.synchronize(device)
    peak = torch.cuda.max_memory_allocated(device)

    kv_heads = getattr(config, "num_key_value_heads", config.num_attention_heads)
    head_dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
    expected = kv_bytes(config.num_hidden_layers, kv_heads, head_dim, FP16_BYTES, length, BATCH_SIZE)
    actual = cache_tensor_bytes(output.past_key_values)
    print_bytes("KV 공식", expected)
    print_bytes("KV 실제", actual)
    if actual != expected:
        raise AssertionError("KV 텐서 바이트가 공식과 일치하지 않습니다.")
    print("KV 검증: 일치")
    print_bytes("전체 로짓 공식 (FP16)", BATCH_SIZE * length * config.vocab_size * FP16_BYTES)
    print_bytes(f"반환 로짓 ({output.logits.dtype})", tensor_bytes(output.logits))
    print_bytes("prefill 최대 총 할당", peak)
    print_bytes("prefill 최대 증가", peak - baseline)
    return peak


def measure_model(repo):
    import transformers
    from transformers import AutoModelForCausalLM

    device = torch.device("cuda:0")
    print(f"GPU: {torch.cuda.get_device_name(device)} | PyTorch: {torch.__version__} | Transformers: {transformers.__version__}")
    print(f"모델: {repo}")

    model = None
    try:
        cleanup(device)
        before = torch.cuda.memory_allocated(device)
        try:
            model = AutoModelForCausalLM.from_pretrained(
                repo,
                dtype=torch.float16,
                attn_implementation="sdpa",
                use_safetensors=True,
                device_map={"": device},
                token=False,
            ).eval()
        except torch.cuda.OutOfMemoryError:
            print("모델을 로드할 CUDA 메모리가 부족합니다.")
            return
        torch.cuda.synchronize(device)
        baseline = torch.cuda.memory_allocated(device)
        parameters = list(model.parameters())
        api_count = API_PARAMETERS[repo]
        expected = weight_bytes(api_count, FP16_BYTES)
        increase = baseline - before
        tied = model.get_input_embeddings().weight is model.get_output_embeddings().weight
        print(f"API 파라미터 수: {api_count:,} | 고유 파라미터 수: {sum(p.numel() for p in parameters):,}")
        print(f"입출력 임베딩 가중치 공유: {'예' if tied else '아니요'}")
        print_bytes("가중치 공식 (API 수 × 2)", expected)
        print_bytes("고유 가중치 텐서", sum(map(tensor_bytes, parameters)))
        del parameters
        print_bytes("가중치 로드 증가", increase)
        print_bytes("로드 증가 − 공식", increase - expected)
        print_bytes("로드 후 기준선", baseline)

        for length in LENGTHS:
            peaks = []
            for keep, mode in ((0, "전체 로짓"), (1, "마지막 토큰 로짓")):
                print(f"\n{length:,}토큰 | {mode} (logits_to_keep={keep})")
                peak = None
                try:
                    peak = measure_prefill(model, device, length, keep, baseline)
                except torch.cuda.OutOfMemoryError:
                    print("prefill에 필요한 CUDA 메모리가 부족합니다.")
                finally:
                    cleanup(device)
                peaks.append(peak)
            all_peak, last_peak = peaks
            print_bytes("최대 할당 차이 (전체 − 마지막)", None if None in peaks else all_peak - last_peak)
    finally:
        del model
        cleanup(device)


if torch.cuda.is_available():
    measure_model(MODEL_REPO)
else:
    print("CUDA가 없어 T4 실측을 건너뜁니다.")

## 결과 읽기

1절에서 저장 용량 대비 가중치 계산값의 차이를 확인합니다. 3절에서는 로드 증가량을 `API 파라미터 수 × 2`와, `KV 실제`를 `KV 공식`과 비교합니다. 가중치 차이가 보이면 고유 파라미터 수와 입출력 임베딩 공유 여부도 확인하세요.

가장 긴 입력의 전체 로짓 실행에서 `반환 로짓`과 `KV 실제`를 비교하면 로짓 텐서가 KV 캐시보다 큰지 확인할 수 있습니다. 같은 입력의 `최대 할당 차이 (전체 − 마지막)`로 로짓 반환 범위를 줄였을 때의 효과를 확인하세요.